# Solutions 07: On-Policy Distillation

This notebook solves the four exercises of Lab 07 (`labs/lab-07-onpolicy-distillation.ipynb`).
Execution status: exercises 3 and 4 run fully live (exercise 3 is pure replay of logged
trajectories, exercise 4 is a toy policy under 50 optimizer steps). Exercises 1 and 2 carry
live demonstrations of the mechanism plus full GKD runs gated behind `RUN_TRAINING = False`,
the lab's own Tier 2 pattern.

Attempt the exercises before reading this. In particular, try to predict exercise 2's answer
from Lab 05 before looking; the point of that exercise is the prediction, not the run.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, dataclasses
sys.path.insert(0, "../code")
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"

import torch
import torch.nn.functional as F

from kd_core import mean_entropy, onpolicy_mask, kl_divergence
from kd_pipeline import set_seed_everywhere, config_fingerprint, EntropyMonitor, RunManifest

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Exercise 1: `lmbda` fine-grain

**The exercise, restated.** Add `lmbda` = 0.25 and 0.75 to the lab's {0.0, 0.5, 1.0} sweep
(`lmbda` is the fraction of training batches generated by the student itself rather than
taken from the fixed corpus). The GKD paper reports task-dependent optima; find where the
generation-quality curve bends on this task.

**The approach.** The two extra training runs go behind the flag. What can run live is the
part people usually skip: deciding *in advance* what curve shape would answer the question,
so that when the five points come back there is a mechanical reading of them instead of a
squint. Two hypotheses cover the paper's findings:

- *H-monotone:* more on-policy is simply better here, so quality rises with `lmbda` and the
  best point is the boundary at 1.0. The curve has no interior bend.
- *H-interior:* on-policy data helps until the loss of fixed-corpus grounding hurts more, so
  the curve rises, bends, and falls, with the best point strictly inside (0, 1).

The discriminator is the discrete second difference (the change of the change between
adjacent grid points): an interior optimum shows up as an argmax away from both endpoints
plus a strongly negative second difference at the bend. The live cell builds one synthetic
curve per hypothesis, runs that detector, and asserts it separates them. It also builds the
five arms' *entropy* trajectories the way the lab's Part C describes healthy runs (deeper
decline the more on-policy the arm) and asserts the lab's `EntropyMonitor` stays quiet on
all of them, because a fine-grained `lmbda` sweep is only interpretable if none of its arms
tripped the babysitter.

In [2]:
# Live: the curve-shape referee, then the healthy-entropy check for a 5-point sweep.
GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

# Synthetic generation-quality curves (teacher-scored rollout quality, arbitrary
# units) for the two hypotheses. These are stand-ins for the gated runs' outputs.
H_monotone = [50.0, 54.0, 57.0, 59.0, 60.5]            # rises to the boundary
H_interior = [50.0, 56.0, 60.0, 59.0, 55.0]            # bends at 0.5

def read_curve(vals):
    best = GRID[max(range(len(vals)), key=lambda i: vals[i])]
    second = [vals[i-1] - 2*vals[i] + vals[i+1] for i in range(1, len(vals)-1)]
    bend = GRID[1 + min(range(len(second)), key=lambda i: second[i])]
    return best, bend, second

for name, vals in (("H-monotone", H_monotone), ("H-interior", H_interior)):
    best, bend, second = read_curve(vals)
    print(f"{name}: best lmbda {best}, sharpest bend at {bend}, "
          f"second differences {[round(s,1) for s in second]}")

best_m, _, sec_m = read_curve(H_monotone)
best_i, bend_i, sec_i = read_curve(H_interior)
assert best_m == 1.0, "a monotone curve must put its best point on the boundary"
assert 0.0 < best_i < 1.0 and bend_i == best_i, \
    "an interior optimum must show an interior argmax at the sharpest bend"
assert min(sec_i) < min(sec_m) - 2.0, "the interior bend must be visibly sharper"

# Entropy trajectories for the 5-arm sweep, shaped per Part C: healthy decline,
# deeper the more on-policy the arm. The babysitter must stay quiet on all five.
g = torch.Generator().manual_seed(SEED)
quiet = True
for lam in GRID:
    mon = EntropyMonitor(floor_nats=0.15, drop_frac=0.6, window=8)
    for s in range(0, 601, 25):
        depth = 0.2 + 0.6 * lam                       # on-policy commits harder
        h = 3.4 - depth * (1 - math.exp(-s / 300)) + 0.015 * torch.randn(1, generator=g).item()
        mon.update(s, h)
    print(f"lmbda {lam:4.2f}: {mon.report()}")
    quiet &= not mon.collapsed
assert quiet, "every healthy arm must leave the monitor quiet, or the sweep is uninterpretable"
print("\ncurve referee calibrated; a 5-point sweep with quiet monitors is readable by machine")

H-monotone: best lmbda 1.0, sharpest bend at 0.25, second differences [-1.0, -1.0, -0.5]
H-interior: best lmbda 0.5, sharpest bend at 0.5, second differences [-2.0, -5.0, -3.0]
lmbda 0.00: entropy 3.379 nats @ step 0 -> 3.249 @ step 600 (healthy)
lmbda 0.25: entropy 3.408 nats @ step 0 -> 3.108 @ step 600 (healthy)
lmbda 0.50: entropy 3.379 nats @ step 0 -> 2.954 @ step 600 (healthy)
lmbda 0.75: entropy 3.405 nats @ step 0 -> 2.857 @ step 600 (healthy)
lmbda 1.00: entropy 3.392 nats @ step 0 -> 2.699 @ step 600 (healthy)

curve referee calibrated; a 5-point sweep with quiet monitors is readable by machine


In [3]:
# Gated: the two extra GKD runs, identical to the lab's run_arm with new lmbda.
if RUN_TRAINING:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl.experimental.gkd import GKDConfig, GKDTrainer
    from datasets import Dataset

    def run_lmbda(lam):
        set_seed_everywhere(SEED)
        tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
        student = AutoModelForCausalLM.from_pretrained(
            "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.bfloat16)
        teacher = AutoModelForCausalLM.from_pretrained(
            "HuggingFaceTB/SmolLM2-1.7B-Instruct", dtype=torch.bfloat16)
        tr = torch.load("../data/lab03/train.pt")
        msgs = []
        for i in range(1024):
            text = tok.decode(tr["input_ids"][i, :tr["prompt_lens"][i]],
                              skip_special_tokens=True)
            msgs.append({"messages": [{"role": "user", "content": text},
                                      {"role": "assistant", "content": ""}]})
        args = GKDConfig(
            output_dir=f"../runs/lab07/lmbda-{lam}",
            lmbda=lam, beta=0.5, temperature=0.9, max_new_tokens=128,
            per_device_train_batch_size=4, gradient_accumulation_steps=8,
            learning_rate=3e-5, max_steps=600, logging_steps=25, bf16=True,
            report_to=[],
        )
        trainer = GKDTrainer(model=student, teacher_model=teacher, args=args,
                             processing_class=tok,
                             train_dataset=Dataset.from_list(msgs))
        trainer.train()
        trainer.save_model(f"../runs/lab07/lmbda-{lam}/final")
        del student, teacher, trainer

    for lam in (0.25, 0.75):
        run_lmbda(lam)
    # Then score all five arms' rollouts with the teacher (sequence log-prob on
    # held-out prompts) and feed the 5-point curve to read_curve() above.
else:
    print("RUN_TRAINING=False: the 0.25 and 0.75 arms compiled but did not execute.")

RUN_TRAINING=False: the 0.25 and 0.75 arms compiled but did not execute.


**Interpretation.** The live cell shows the referee working: on the monotone curve the
detector reports the best point on the boundary, and on the interior curve it reports the
argmax strictly inside the grid, sitting exactly at the sharpest (most negative) second
difference, with the asserted gap between the two shapes. The entropy check passed for all
five synthetic arms, which is the precondition the real sweep must also meet: a `lmbda`
point whose monitor tripped is not a data point, it is a crashed run wearing one.

The two new training runs are gated (`RUN_TRAINING=False`; the curves above are synthetic
stand-ins, not results). Expected outcome, grounded in the lab's Part C: quality measured on
the student's *own rollouts scored by the teacher* should rise from `lmbda` 0.0 through 0.5,
since that gap is exposure bias (training only on others' tokens, then having to condition
on your own) closing. Between 0.75 and 1.0 the GKD paper's task dependence shows up: on
tasks like this one, where the fixed corpus matches the eval prompts well, expect the curve
to flatten or bend slightly down at 1.0, because pure on-policy gives up the corpus's
grounding entirely; an interior best point at 0.5 to 0.75 is the likely reading. Confirming
H-monotone instead is a legitimate result and would say corpus grounding contributes nothing
here. Failure signatures: a sawtooth curve (points not comparable, usually different
effective step counts after a monitor stop), and teacher-forced agreement moving opposite to
rollout quality across the grid, which is not a contradiction but the lab's whole point:
teacher-forced metrics cannot see what on-policy training fixes.

## Exercise 2: The beta times on-policy interaction

**The exercise, restated.** Rerun the `on` arm with beta = 0, meaning forward KL (the
divergence KL(teacher || student), which punishes the student for assigning near-zero
probability anywhere the teacher assigns mass) computed on the student's own rollouts.
Theory calls this the odd combination. What actually happens to entropy?

**The approach.** The run is gated; the oddity itself is demonstrable with a few
distributions. Two live demonstrations pin it down:

1. *Scoring sampled tokens under both directions.* When you only get to evaluate log
   probabilities at tokens the student actually sampled, the two directions are not
   symmetric. The average of log q minus log p over the student's own samples *is* reverse
   KL, an unbiased estimate by definition, because reverse KL is an expectation under the
   student. Forward KL is an expectation under the *teacher*, so estimating it from student
   samples needs importance weights p over q, and those weights are astronomically small
   exactly where the student has abandoned a teacher mode, so the estimate silently misses
   the one region forward KL exists to punish. The cell builds a teacher with a mode the
   student never samples and shows the sampled estimate of forward KL missing most of the
   true value while the reverse estimate lands on its closed form. (GKD itself dodges this
   estimator problem: the teacher returns its full distribution at each rollout position, so
   the per-position forward KL is computed exactly over the vocabulary. What stays
   student-driven is *which positions exist at all*, the state distribution.)
2. *What each direction does to entropy under a capacity limit.* A small student cannot
   match a spread-out teacher exactly. Restrict a categorical student to 2 degrees of
   freedom against a bimodal 10-token teacher and train it both ways: forward KL must cover
   both modes (high entropy), reverse KL locks onto one (near-zero entropy). This is Lab
   05's beta column reproduced in ten lines, and it is the direction the entropy answer
   comes from.

In [4]:
# Live demo 1: scoring sampled tokens under both directions.
set_seed_everywhere(SEED)
V = 10
p_logits = torch.full((V,), 0.0)
p_logits[0] = 3.0                                   # a strong teacher mode at token 0
p_log = F.log_softmax(p_logits, -1)

q_probs = F.softmax(torch.full((V,), 0.0), -1).clone()
q_probs[0] = 1e-9                                   # the student has abandoned that mode
q_probs = q_probs / q_probs.sum()
q_log = q_probs.log()

fwd_true = float((p_log.exp() * (p_log - q_log)).sum())      # KL(p || q), closed form
rev_true = float((q_probs * (q_log - p_log)).sum())          # KL(q || p), closed form

n = 5000
x = torch.multinomial(q_probs, n, replacement=True)          # student rollout tokens
rev_est = float((q_log[x] - p_log[x]).mean())                # E_q[log q - log p]
w = (p_log[x] - q_log[x]).exp()                              # importance weights p/q
fwd_est = float((w * (p_log[x] - q_log[x])).mean())          # E_q[(p/q) log(p/q)]

print(f"forward KL(p||q): closed form {fwd_true:7.3f} | from student samples {fwd_est:7.3f}")
print(f"reverse KL(q||p): closed form {rev_true:7.3f} | from student samples {rev_est:7.3f}")
assert abs(rev_est - rev_true) / rev_true < 0.10, "reverse KL is estimable from own samples"
assert fwd_est < 0.5 * fwd_true, \
    "the sampled forward estimate must miss the never-sampled teacher mode"
assert int((x == 0).sum()) == 0, "the abandoned mode was indeed never sampled"

# Live demo 2: entropy under each direction with a capacity-limited student.
p2 = torch.full((V,), -4.0); p2[1] = 2.0; p2[8] = 2.0        # bimodal teacher
p2_log = F.log_softmax(p2, -1)
W = torch.randn(V, 2, generator=torch.Generator().manual_seed(SEED))

def train_direction(direction, steps=40):
    z = torch.zeros(2, requires_grad=True)                    # 2 degrees of freedom only
    opt = torch.optim.Adam([z], lr=0.3)
    for _ in range(steps):
        ql = F.log_softmax(W @ z, -1)
        loss = ((p2_log.exp() * (p2_log - ql)).sum() if direction == "forward"
                else (ql.exp() * (ql - p2_log)).sum())
        opt.zero_grad(); loss.backward(); opt.step()
    ql = F.log_softmax(W @ z, -1).detach()
    return float(-(ql.exp() * ql).sum())

H_fwd, H_rev = train_direction("forward"), train_direction("reverse")
print(f"\ncapacity-limited student vs bimodal teacher, 40 steps each:")
print(f"  trained with forward KL: entropy {H_fwd:.3f} nats (covers both modes)")
print(f"  trained with reverse KL: entropy {H_rev:.3f} nats (commits to one)")
assert H_fwd > H_rev + 0.5, "forward KL must leave the student visibly more spread out"
print("\nboth demos hold: forward KL is the mass-covering direction, and it is exactly")
print("the direction you cannot estimate from the student's own samples alone")

forward KL(p||q): closed form  13.619 | from student samples  -0.363
reverse KL(q||p): closed form   1.173 | from student samples   1.173



capacity-limited student vs bimodal teacher, 40 steps each:
  trained with forward KL: entropy 1.476 nats (covers both modes)
  trained with reverse KL: entropy 0.090 nats (commits to one)

both demos hold: forward KL is the mass-covering direction, and it is exactly
the direction you cannot estimate from the student's own samples alone


In [5]:
# Gated: the real rerun, the lab's `on` arm with beta=0.0.
if RUN_TRAINING:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl.experimental.gkd import GKDConfig, GKDTrainer
    from datasets import Dataset

    set_seed_everywhere(SEED)
    tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
    student = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.bfloat16)
    teacher = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B-Instruct", dtype=torch.bfloat16)
    tr = torch.load("../data/lab03/train.pt")
    msgs = []
    for i in range(1024):
        text = tok.decode(tr["input_ids"][i, :tr["prompt_lens"][i]],
                          skip_special_tokens=True)
        msgs.append({"messages": [{"role": "user", "content": text},
                                  {"role": "assistant", "content": ""}]})
    args = GKDConfig(
        output_dir="../runs/lab07/on-beta0",
        lmbda=1.0, beta=0.0,                 # forward KL on the student's own rollouts
        temperature=0.9, max_new_tokens=128,
        per_device_train_batch_size=4, gradient_accumulation_steps=8,
        learning_rate=3e-5, max_steps=600, logging_steps=25, bf16=True, report_to=[],
    )
    trainer = GKDTrainer(model=student, teacher_model=teacher, args=args,
                         processing_class=tok, train_dataset=Dataset.from_list(msgs))
    trainer.train()          # attach the lab's RolloutWatch callback to log entropy
    trainer.save_model("../runs/lab07/on-beta0/final")
else:
    print("RUN_TRAINING=False: the beta=0 on-policy arm compiled but did not execute.")

RUN_TRAINING=False: the beta=0 on-policy arm compiled but did not execute.


**Interpretation.** The first demo's printed numbers carry the whole argument: the reverse
KL estimated from the student's own 5000 sampled tokens landed within the asserted 10
percent of its closed form, while the sampled forward-KL estimate came out near zero (in
this run slightly negative, -0.36 against a true 13.6), because the entire divergence lives
on a token the student sampled zero times and the surviving sampled terms are small
log-ratios of the wrong sign. The second demo gives the
entropy direction with an asserted gap of more than 0.5 nats: forward KL trained the
capacity-limited student to 1.5 nats of spread, reverse KL to 0.05.

So what should the gated rerun show (`RUN_TRAINING=False` here, one line of honesty)?
Expected result, grounded in Lab 05's beta column and the lab's Part C: the beta=0 on-policy
arm's rollout entropy should decline *less* than the beta=0.5 arm's, plateauing near the
teacher's own spread, and the `EntropyMonitor` should stay far from tripping, because
forward KL pushes the student to keep mass everywhere the teacher does, at every state the
rollouts visit. The combination is odd, not unstable: it spends on-policy generation (whose
purpose is letting the student concentrate on its own reachable behavior) on an objective
that forbids concentrating. The cost shows up as quality, not as collapse: expect
teacher-scored rollout quality below the beta=0.5 arm at equal steps, the smeared-student
effect, since a 360M student made to cover a 1.7B teacher's spread at its own visited states
has less probability left for the tokens it can actually get right. Refuting evidence would
be entropy *collapsing* under beta=0; that signature would point at the state-distribution
feedback loop (narrow rollouts beget narrow training positions) overpowering the
mass-covering loss, and it would be worth a seed-replication before believing.

## Exercise 3: The monitor calibration curve

**The exercise, restated.** Sweep the `EntropyMonitor`'s `drop_frac` (the fraction of
entropy that may be lost within the trailing window before the monitor flags collapse) over
{0.3, 0.45, 0.6} on the five arms' logged entropy trajectories. No retraining: replay the
histories. Which setting catches the `degenerate` arm earliest without false-firing on `on`?

**The approach.** This is a detector-calibration problem, and the honest way to do it is the
way the lab tested the monitor in Part A·2: build trajectories with known ground truth, then
measure two numbers per threshold, detection latency on the arm that truly collapses and
false fires on the arms that do not. Since the real Part B logs did not execute in this
environment, I construct the five trajectories to the exact shapes the lab's Part C
describes, logged every 25 steps over 600 steps like the lab's `logging_steps=25`,
`max_steps=600` configuration, with seeded noise:

- `off`: gentle decline, 3.4 to 3.0 nats, plateauing. Healthy by construction.
- `mixed`: the same shape, a little deeper (to about 2.7), still healthy.
- `on`: the cold-start arm. Part C says entropy "jumps around in the first ~50 steps"
  because a base student's rollouts are garbage and the teacher's scores on garbage are
  noise; so the first eight observations carry swings of up to 0.9 nats around the trend
  before settling into a healthy decline. Healthy ground truth, but a trap for a twitchy
  threshold.
- `on-warm`: starts lower (the distilled init has already committed) and declines calmly.
- `degenerate`: flat until step 300, then an accelerating exponential collapse with a time
  constant of 120 steps, the Part C signature of "usually late, usually after things looked
  fine". Ground truth: collapsed.

Then the sweep: one fresh monitor per (threshold, arm) pair, window 8 observations and floor
0.15 nats exactly as Part B configures it, replay, record the first step at which
`collapsed` goes true. The claims to assert: every threshold catches `degenerate`; detection
step is monotone in `drop_frac` (a looser threshold fires later); 0.3 false-fires on the
cold-start jitter; 0.45 and 0.6 have zero false fires.

In [6]:
# Live: build the five trajectories, shaped to Part C's descriptions.
set_seed_everywhere(SEED)
g = torch.Generator().manual_seed(SEED)
STEPS = list(range(0, 601, 25))                    # the lab logs every 25 of 600 steps
COLD_JITTER = [0.0, 0.8, -0.6, 0.5, -0.9, 0.3, -0.2, 0.1]   # first ~175 steps of `on`

def trajectory(arm):
    vals = []
    for i, s in enumerate(STEPS):
        if arm == "off":
            h = 3.4 - 0.4 * (1 - math.exp(-s / 300))
        elif arm == "mixed":
            h = 3.4 - 0.7 * (1 - math.exp(-s / 300))
        elif arm == "on":
            h = 3.5 - 1.0 * (1 - math.exp(-s / 350))
            h += COLD_JITTER[i] if i < len(COLD_JITTER) else 0.0
        elif arm == "on-warm":
            h = 2.9 - 0.3 * (1 - math.exp(-s / 250))
        elif arm == "degenerate":
            h = 3.3 - 0.0002 * s if s < 300 else 3.24 * math.exp(-(s - 300) / 120)
        vals.append(h + 0.015 * torch.randn(1, generator=g).item())
    return vals

ARMS = ["off", "mixed", "on", "on-warm", "degenerate"]
TRAJ = {a: trajectory(a) for a in ARMS}
for a in ARMS:
    v = TRAJ[a]
    print(f"{a:>10}: start {v[0]:5.2f}  step300 {v[12]:5.2f}  end {v[-1]:5.2f} nats")
assert TRAJ["degenerate"][-1] < 0.5 < TRAJ["off"][-1], "ground truth shapes as intended"
assert max(TRAJ["on"][:8]) - min(TRAJ["on"][:8]) > 1.0, "cold-start jitter present"
print("\nfive trajectories built; ground truth: only `degenerate` collapses")

       off: start  3.38  step300  3.13  end  3.08 nats
     mixed: start  3.41  step300  2.96  end  2.80 nats
        on: start  3.48  step300  2.91  end  2.67 nats
   on-warm: start  2.91  step300  2.69  end  2.65 nats
degenerate: start  3.29  step300  3.24  end  0.26 nats

five trajectories built; ground truth: only `degenerate` collapses


In [7]:
# Live: the calibration sweep, replayed with the lab's own monitor settings.
HEALTHY = ["off", "mixed", "on", "on-warm"]

def replay(vals, drop_frac):
    mon = EntropyMonitor(floor_nats=0.15, drop_frac=drop_frac, window=8)
    for s, h in zip(STEPS, vals):
        mon.update(s, h)
        if mon.collapsed:
            return s
    return None

results = {}
print(f"{'drop_frac':>9} | " + " ".join(f"{a:>10}" for a in ARMS) +
      f" | {'det. step':>9} {'false fires':>11}")
for df in (0.30, 0.45, 0.60):
    row = {a: replay(TRAJ[a], df) for a in ARMS}
    det = row["degenerate"]
    false_fires = [a for a in HEALTHY if row[a] is not None]
    results[df] = (det, false_fires)
    cols = " ".join(f"{str(row[a] if row[a] is not None else '-'):>10}" for a in ARMS)
    print(f"{df:>9.2f} | {cols} | {det:>9} {len(false_fires):>11}")

det30, ff30 = results[0.30]
det45, ff45 = results[0.45]
det60, ff60 = results[0.60]
assert det30 is not None and det45 is not None and det60 is not None, \
    "every threshold must catch the true collapse"
assert det30 < det45 < det60, "a tighter threshold must detect earlier"
assert ff30 == ["on"], "0.30 must false-fire on the cold-start jitter and nothing else"
assert ff45 == [] and ff60 == [], "0.45 and 0.60 must stay quiet on all healthy arms"
assert det45 - det30 <= 50, "0.45 gives up at most two logging intervals of latency"
print("\ncalibration verdict: drop_frac 0.45 catches the collapse at step "
      f"{det45} (0.30: {det30} but with a false fire; 0.60: {det60}) with zero false fires")

drop_frac |        off      mixed         on    on-warm degenerate | det. step false fires
     0.30 |          -          -        100          -        350 |       350           1
     0.45 |          -          -          -          -        375 |       375           0
     0.60 |          -          -          -          -        425 |       425           0

calibration verdict: drop_frac 0.45 catches the collapse at step 375 (0.30: 350 but with a false fire; 0.60: 425) with zero false fires


**Interpretation.** The table is the exercise's answer, and the asserts pin its three
facts. First, all three thresholds catch the real collapse, but with different latency: 0.30
at step 350, 0.45 at step 375, 0.60 not until step 425, which is monotone the way a
fraction-of-drop rule must be (a looser threshold needs more of the collapse to have already
happened, and with an exponential collapse each extra 25-step interval forfeits about
another fifth of the remaining entropy). Second, the tight threshold is not free: 0.30
false-fired on the `on` arm at step 100, exactly on the cold-start jitter Part C warned
about, where a 0.9-nat swing inside the 8-observation window looks like a 30 percent drop
because it is one, just not a persistent one. A monitor that halts a healthy cold-start run
at step 100 costs you the entire run, which is a worse failure than detecting a collapse two
intervals late, since the collapse-detection purpose is saving the last good checkpoint.
Third, 0.45 and 0.60 both kept a clean record on the healthy arms, so between them the
latency argument decides: 0.45 saves two logging intervals (50 steps) of overwritten
checkpoints over the lab's default 0.60.

So the answer to "which setting" on these trajectories: **0.45**, with the lab's window of 8
and floor of 0.15 unchanged. Two caveats belong in the verdict. These are constructed
trajectories, built to Part C's described shapes and seeded, not the gated Part B's real
logs; the calibration *procedure* is the deliverable, and rerunning this cell on real
`EntropyMonitor.history` lists is a copy-paste. And the 0.30-vs-jitter collision points at a
better monitor rather than a better threshold: exempting the first N observations, or
comparing against a short median instead of the window's first element, would let a tight
threshold coexist with cold starts. That the calibration sweep *found* that design pressure
is exactly why the lab makes you run one.

## Exercise 4: The rollout buffer question

**The exercise, restated.** `GKDTrainer` regenerates rollouts every step; older
reinforcement-learning practice collects a buffer of rollouts and reuses it for several
epochs (several full passes) before refreshing. Implement 2-epoch reuse and measure the
staleness cost, the quality lost because the buffer's rollouts came from an earlier version
of the student.

**The approach.** The real GKD variant goes behind the flag. The staleness mechanism itself
fits in a toy small enough to run in seconds, provided the toy keeps the one structural
ingredient that makes staleness *mean* something: a state distribution that the policy
controls. So the toy policy generates two-token sequences over a 12-token vocabulary: a
learnable initial distribution picks the first token (the "state"), and a learnable
per-state conditional picks the second. The teacher is a fixed pair of the same shape, with
its initial mass concentrated on states the student initially never visits. Training
minimizes forward KL against the teacher at position one, plus forward KL at position two
*at the states appearing in the batch*, which is exactly GKD's shape: the per-state loss is
computed exactly, but which states get trained is decided by whoever generated the batch.

Fresh training samples states from the current student every step. Buffered training samples
64 states at once and consumes them in 16-state minibatches for 2 epochs, refreshing every 8
steps, so its gradients weight states by where the student *was* up to 8 steps ago. Same
optimizer, same step count (40), same seeds; the only difference is buffer staleness, and
generation cost is halved (one 64-sample refresh serves 8 steps instead of 8 fresh draws of
16... the same 128 sampled states serve twice the steps). The final score is the on-policy
divergence: position-one KL plus the expected position-two KL under where the *final*
student actually goes. Three seeds, and the assert requires the stale variant to end worse
on every one.

In [8]:
# Live: 2-epoch buffer reuse vs fresh rollouts on a toy two-position policy.
V = 12
def kl_row(p_log, q_log):
    return (p_log.exp() * (p_log - q_log)).sum(-1)

def run_toy(seed, epochs, steps=40, batch=16, buf_size=64, lr=0.5):
    g = torch.Generator().manual_seed(seed)
    t0 = torch.full((V,), -3.0); t0[0] = 2.5; t0[1] = 2.0     # teacher visits states 0,1
    p0_log = F.log_softmax(t0, -1)
    pc_log = F.log_softmax(
        3.0 * torch.randn(V, V, generator=torch.Generator().manual_seed(7)), -1)
    s0 = torch.full((V,), -3.0); s0[V-1] = 2.5; s0[V-2] = 2.0  # student starts at 10,11
    q0 = s0.clone().requires_grad_(True)
    qc = torch.zeros(V, V, requires_grad=True)
    opt = torch.optim.SGD([q0, qc], lr=lr)
    buf, ptr = None, 0
    refresh = (buf_size // batch) * epochs                     # 8 steps for 2 epochs
    for step in range(steps):
        if epochs == 0:                                        # fresh: sample every step
            with torch.no_grad():
                A = torch.multinomial(F.softmax(q0, -1), batch,
                                      replacement=True, generator=g)
        else:                                                  # buffered: reuse for epochs
            if step % refresh == 0:
                with torch.no_grad():
                    buf = torch.multinomial(F.softmax(q0, -1), buf_size,
                                            replacement=True, generator=g)
                order = torch.cat([torch.randperm(buf_size, generator=g)
                                   for _ in range(epochs)])
                buf, ptr = buf[order], 0
            A = buf[ptr:ptr + batch]; ptr += batch
        loss = (kl_row(p0_log, F.log_softmax(q0, -1))
                + kl_row(pc_log[A], F.log_softmax(qc, -1)[A]).mean())
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        q0_log = F.log_softmax(q0, -1)
        qc_log = F.log_softmax(qc, -1)
        onpolicy_div = float(kl_row(p0_log, q0_log)
                             + (q0_log.exp() * kl_row(pc_log, qc_log)).sum())
    return onpolicy_div

print(f"{'seed':>4} {'fresh':>8} {'2-epoch buffer':>15} {'staleness cost':>15}")
gaps = []
for seed in (0, 1, 2):
    fresh = run_toy(seed, epochs=0)
    stale = run_toy(seed, epochs=2)
    gaps.append(stale - fresh)
    print(f"{seed:>4} {fresh:>8.4f} {stale:>15.4f} {stale - fresh:>15.4f} "
          f"({(stale/fresh - 1)*100:+.0f}%)")

assert all(gap > 0.01 for gap in gaps), \
    "2-epoch reuse must end measurably worse than fresh rollouts on every seed"
print(f"\nmean staleness cost {sum(gaps)/len(gaps):.4f} nats of on-policy divergence,")
print("bought with half the rollout generation: the boundary made continuous")

seed    fresh  2-epoch buffer  staleness cost


   0   0.4261          0.4807          0.0546 (+13%)


   1   0.4475          0.4959          0.0483 (+11%)


   2   0.4285          0.4682          0.0398 (+9%)

mean staleness cost 0.0476 nats of on-policy divergence,
bought with half the rollout generation: the boundary made continuous


In [9]:
# Gated: the real 2-epoch variant, a GKDTrainer subclass that reuses each
# generated on-policy batch once before regenerating.
if RUN_TRAINING:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from trl.experimental.gkd import GKDConfig, GKDTrainer
    from datasets import Dataset
    import random as _random

    class BufferedGKDTrainer(GKDTrainer):
        REUSE = 2                                   # epochs per generated batch

        def __init__(self, *a, **kw):
            super().__init__(*a, **kw)
            self._cached, self._uses = None, 0

        def training_step(self, model, inputs, num_items_in_batch=None):
            if _random.random() <= self.lmbda:
                if self._cached is None or self._uses >= self.REUSE:
                    from trl.models.utils import unwrap_model_for_generation
                    with unwrap_model_for_generation(
                            model, self.accelerator,
                            generation_kwargs=self.generation_kwargs) as um:
                        self._cached = self.generate_on_policy_outputs(
                            um, inputs, self.generation_config)
                    self._uses = 0
                ids, attn, labels = self._cached
                self._uses += 1
                inputs = {**inputs, "input_ids": ids,
                          "attention_mask": attn, "labels": labels}
            # skip GKDTrainer.training_step's own generation by calling its parent
            return super(GKDTrainer, self).training_step(
                model, inputs, num_items_in_batch)

    set_seed_everywhere(SEED)
    tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
    student = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-360M-Instruct", dtype=torch.bfloat16)
    teacher = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B-Instruct", dtype=torch.bfloat16)
    tr = torch.load("../data/lab03/train.pt")
    msgs = []
    for i in range(1024):
        text = tok.decode(tr["input_ids"][i, :tr["prompt_lens"][i]],
                          skip_special_tokens=True)
        msgs.append({"messages": [{"role": "user", "content": text},
                                  {"role": "assistant", "content": ""}]})
    args = GKDConfig(output_dir="../runs/lab07/on-buffered",
                     lmbda=1.0, beta=0.5, temperature=0.9, max_new_tokens=128,
                     per_device_train_batch_size=4, gradient_accumulation_steps=8,
                     learning_rate=3e-5, max_steps=600, logging_steps=25,
                     bf16=True, report_to=[])
    trainer = BufferedGKDTrainer(model=student, teacher_model=teacher, args=args,
                                 processing_class=tok,
                                 train_dataset=Dataset.from_list(msgs))
    trainer.train()
    trainer.save_model("../runs/lab07/on-buffered/final")
else:
    print("RUN_TRAINING=False: the buffered GKD variant compiled but did not execute.")

RUN_TRAINING=False: the buffered GKD variant compiled but did not execute.


**Interpretation.** The toy's table shows the staleness bias appearing, and the assert
holds it to a standard: on all three seeds, the 2-epoch buffer ends with a higher on-policy
divergence than fresh sampling (a gap of 0.04 to 0.05 nats here, 9 to 13 percent in
relative terms) at identical step counts, identical losses, and identical optimizers. The
mechanism is visible in the construction: during the 40 steps the student's initial
distribution migrates from states 10 and 11 to the teacher's states 0 and 1, and the
buffered variant keeps spending conditional-position gradient on states its policy is
already leaving, up to 8 steps behind, so the states its *final* policy actually visits are
under-trained exactly where the on-policy metric looks. That is exposure bias re-entering
through the buffer door: reused rollouts are off-policy data wearing an on-policy label, and
"2 epochs" is a point on a dial between the lab's `off` and `on` arms, not a separate
method.

The price paid for the bias is the point: the buffered run drew half as many rollout
samples. Whether that trade wins depends on what generation costs, which is Lab 08's
subject; on hardware where student decode is cheap, GKD's regenerate-every-step default is
the right side of the trade, and the toy quantifies what the other side loses. The gated
variant (`RUN_TRAINING=False`; unexecuted) makes the same comparison for real. Expected
result, grounded in Part C: the 2-epoch arm lands between `off` and `on` on teacher-scored
rollout quality, closer to `on`, with roughly 35 to 45 percent less wall-clock spent in
generation; entropy trajectories should look like `on` with slightly delayed decline.
Failure signature worth watching there: if the buffered arm ever *beats* fresh on rollout
quality, suspect the eval prompts overlap the buffer refresh boundary (stale rollouts
regularizing an overfitting run is a real effect, but at 600 steps on 1024 prompts,
overfitting is the likelier explanation than magic).